# Supplementary Figure 1 — extended benchmark and recovery detail

Each panel loads its own precomputed CSV output and builds its plot
natively in this notebook (not a rendered PNG from another notebook), so
panels compose at a consistent scale and font size without raster
stretching — same convention as `fig2.ipynb`.

## Setup

In [ ]:
suppressPackageStartupMessages({
  library(data.table)
  library(ggplot2)
  library(patchwork)
  library(cowplot)
  library(yaml)
  library(here)
})


In [ ]:
AXIS_TITLE_SIZE  <- 10.5
AXIS_LABEL_SIZE  <- 8
PLOT_TITLE_SIZE  <- 9
LEGEND_TEXT_SIZE <- 7
LEGEND_TITLE_SIZE <- 8

LABEL_SIZE <- 10
LABEL_FONTFACE <- "bold"
LABEL_FONTFAMILY <- "Helvetica"

# Same font sizes/theme as fig2.ipynb, so both figures read as one paper.
DATASET_LABELS <- c(
  Brain_Mathys2023 = "Brain Mathys",
  Brain_Xiong2023  = "Brain Xiong",
  Heart_Datar2026  = "Heart Datar",
  PBMC_1k1k        = "PBMC 1k1k",
  PBMC_Perez2022   = "PBMC Perez",
  Lung_Sikkema2023 = "Lung Sikkema"
)

# Same 6-dataset -> 4-tissue grouping used natively in 00_benchmark.ipynb's
# by-tissue panel (Brain and PBMC each pool 2 datasets; Heart/Lung are
# singleton tissues).
TISSUE_MAP <- c(
  Brain_Mathys2023 = "Brain", Brain_Xiong2023 = "Brain",
  Heart_Datar2026  = "Heart",
  Lung_Sikkema2023 = "Lung",
  PBMC_1k1k        = "PBMC", PBMC_Perez2022 = "PBMC"
)

# "precursor"/"progenitor" both show up upstream for the same cell type.
abbreviate_ct <- function(x) {
  x <- gsub("Oligodendrocyte [Pp]rogenitor [Cc]ells?", "OPC", x)
  x <- gsub("Oligodendrocyte [Pp]recursor [Cc]ells?", "OPC", x)
  x
}

# Wrap (not truncate) long labels onto multiple lines.
wrap_label <- function(x, width = 14) {
  vapply(x, function(s) paste(strwrap(s, width = width), collapse = "\n"),
         character(1), USE.NAMES = FALSE)
}

theme_pub <- function(base_size = 7, base_line_size = 0.3) {
    theme_classic(base_size = base_size, base_family = "Helvetica") %+replace%
    theme(
        plot.title        = element_text(size = PLOT_TITLE_SIZE, face = "plain", hjust = 0.5,
                                          margin = margin(b = 2)),
        axis.line         = element_line(linewidth = base_line_size),
        axis.ticks        = element_line(linewidth = base_line_size),
        axis.text         = element_text(size = AXIS_LABEL_SIZE),
        axis.title        = element_text(size = AXIS_TITLE_SIZE, face = "plain"),
        legend.text       = element_text(size = LEGEND_TEXT_SIZE),
        legend.title      = element_text(size = LEGEND_TITLE_SIZE, face = "plain"),
        legend.key.size   = unit(3, "mm"),
        legend.background = element_blank(),
        legend.key        = element_blank(),
        panel.grid        = element_blank(),
        strip.text        = element_text(size = PLOT_TITLE_SIZE, face = "plain"),
        strip.background  = element_blank(),
        plot.margin       = unit(c(1, 2, 1, 1), "mm")
    )
}

cfg <- yaml::read_yaml(here("config.yaml"))
MODEL_COLORS_RAW <- unlist(cfg$MODEL_COLORS)
names(MODEL_COLORS_RAW)[names(MODEL_COLORS_RAW) == "GenomicSuperSignature"] <- "GSSig"

# Raw CSVs use abbreviated cell-type codes ("Ast", "Exc") -- same
# code -> display-label lookup 02_disentangle.ipynb / 03_b_matrix_singlecell.ipynb
# / 04_hard_cell_types.ipynb already use natively.
ct_labels_df <- read.csv(here("data", "pseudobulk", "cell_type_labels.csv"), stringsAsFactors = FALSE)
CT_LABELS <- setNames(ct_labels_df$label, ct_labels_df$cell_type)
ct_label <- function(x) ifelse(x %in% names(CT_LABELS), CT_LABELS[x], x)

# Purity (Panel C) has no meaningful negative side -- sequential white->green
# only, reusing the same green hex stops as the diverging scale below.
GREEN_SCALE <- c("white", "#f7fbf7", "#a1d99b", "#007a33")
# Correlation heatmaps (Panels D/E): diverging red/green, weak correlations
# stay near white -- same convention as 02_disentangle.ipynb/04_hard_cell_types.ipynb.
DIVERGING_COLORS <- c("#b2182b", "#fdf7f7", "white", "#f7fbf7", "#007a33")
DIVERGING_VALUES <- scales::rescale(c(-1, -0.5, 0, 0.5, 1))


## Panel A: benchmark boxplots, faceted by tissue

In [ ]:
long <- fread(snakemake@input[["benchmark_long"]])
long_box <- long[truth == "v0" & !is.na(cor)]
long_box[, tissue := TISSUE_MAP[dataset]]

mean_df_tissue <- long_box[, .(mean_cor = mean(cor, na.rm = TRUE),
                                max_cor  = max(cor, na.rm = TRUE)), by = .(tissue, method)]
setorder(mean_df_tissue, tissue, -mean_cor)
mean_df_tissue[, tissue_method := factor(paste(tissue, method, sep = "___"),
                                          levels = paste(tissue, method, sep = "___"))]

long_box_tissue <- merge(long_box, mean_df_tissue[, .(tissue, method, tissue_method)],
                          by = c("tissue", "method"))

METHOD_ORDER_ALL <- long_box[, .(m = mean(cor, na.rm = TRUE)), by = method][order(-m), method]
METHOD_COLORS <- MODEL_COLORS_RAW[METHOD_ORDER_ALL]
METHOD_COLORS[is.na(METHOD_COLORS)] <- "grey70"
names(METHOD_COLORS) <- METHOD_ORDER_ALL

plot_A <- ggplot(long_box_tissue, aes(tissue_method, cor, fill = method)) +
  geom_boxplot(width = 0.6, outlier.shape = NA, color = "black",
               linewidth = 0.3, alpha = 0.85) +
  geom_jitter(width = 0.10, size = 0.5, shape = 21, fill = "white",
              color = "#333333", stroke = 0.2, alpha = 0.7) +
  geom_point(data = mean_df_tissue, aes(x = tissue_method, y = mean_cor), shape = 23,
             size = 1.6, fill = "white", color = "black", stroke = 0.45,
             inherit.aes = FALSE) +
  geom_text(data = mean_df_tissue,
            aes(x = tissue_method, y = max_cor + 0.05, label = sprintf("%.3f", mean_cor)),
            size = 2.6, fontface = "bold", inherit.aes = FALSE) +
  facet_wrap(~tissue, scales = "free", ncol = 2) +
  coord_cartesian(ylim = c(0, NA)) +
  scale_x_discrete(labels = function(x) sub("^.*___", "", x)) +
  scale_fill_manual(values = METHOD_COLORS, na.value = "grey70") +
  labs(x = NULL, y = "Max Pearson r per cell type") +
  theme_pub() +
  theme(legend.position = "none",
        axis.text.x = element_text(angle = 35, hjust = 1, size = AXIS_LABEL_SIZE),
        panel.grid.major = element_line(color = "grey88", linewidth = 0.25),
        panel.grid.minor = element_blank(),
        strip.background = element_rect(fill = "grey94", color = "black", linewidth = 0.3))

options(repr.plot.width = 10, repr.plot.height = 7)
print(plot_A)


## Panel B: bootstrap win rate

In [ ]:
win_rate <- fread(snakemake@input[["bootstrap"]])
setorder(win_rate, -win_rate)
win_rate[, method := factor(method, levels = method)]

plot_B <- ggplot(win_rate, aes(method, win_rate, fill = method)) +
    geom_col(width = 0.6) +
    scale_fill_manual(values = MODEL_COLORS_RAW, na.value = "grey70") +
    scale_y_continuous(labels = function(x) paste0(x * 100, "%"), limits = c(0, 1)) +
    labs(x = NULL, y = "Bootstrap win rate (rank 1)") +
    theme_pub() +
    theme(legend.position = "none",
          axis.text.x = element_text(angle = 35, hjust = 1, size = AXIS_LABEL_SIZE),
          panel.grid.major.y = element_line(color = "grey88", linewidth = 0.25))

options(repr.plot.width = 10, repr.plot.height = 4)
print(plot_B)


## Panel C: per-dataset single-cell recovery (purity) heatmaps

In [ ]:
heatmap_long <- fread(snakemake@input[["heatmap_long"]])

# CLAMPfull-only mean Pearson r per dataset (Panels C/D/E are all
# CLAMPfull-specific analyses) -- reuses long_box already loaded by Panel A.
mean_r_by_dataset <- long_box[method == "CLAMPfull", .(mean_r = mean(cor, na.rm = TRUE)), by = dataset]

DATASETS_C <- names(DATASET_LABELS)

purity_blocks <- lapply(DATASETS_C, function(ds) {
  d <- heatmap_long[dataset == ds]
  if (nrow(d) == 0) return(NULL)

  ct_order <- unique(d$row_cell_type)
  ct_order <- ct_order[order(ct_label(ct_order))]
  ct_labels_ord <- ct_label(ct_order)

  d[, row_label   := factor(ct_label(row_cell_type), levels = ct_labels_ord)]
  d[, col_label   := factor(ct_label(col_cell_type), levels = ct_labels_ord)]
  d[, is_diagonal := row_cell_type == col_cell_type]

  mean_r <- mean_r_by_dataset[dataset == ds, mean_r]
  subtitle_txt <- if (length(mean_r) && !is.na(mean_r)) sprintf("mean r = %.3f", mean_r) else ""

  p <- ggplot(d, aes(x = col_label, y = row_label, fill = pct)) +
    geom_tile(color = "white", linewidth = 0.4) +
    geom_tile(data = d[is_diagonal == TRUE], fill = NA, color = "black", linewidth = 1) +
    geom_text(aes(label = sprintf("%.1f", pct)), size = 2.2) +
    scale_fill_gradientn(colours = GREEN_SCALE, limits = c(0, 100), name = "Top 1%\npurity (%)") +
    coord_fixed() +
    theme_bw(base_size = 7.5) +
    theme(axis.text.x   = element_text(angle = 45, hjust = 1, vjust = 1, size = 6),
          axis.text.y   = element_text(size = 6.5),
          plot.title    = element_text(face = "bold", size = 8, hjust = 0.5),
          plot.subtitle = element_text(size = 6.5, hjust = 0.5, face = "italic")) +
    labs(x = NULL, y = NULL, title = DATASET_LABELS[ds], subtitle = subtitle_txt)

  list(plot = p, n_cat = length(ct_order))
})
names(purity_blocks) <- DATASETS_C
purity_blocks <- Filter(Negate(is.null), purity_blocks)

purity_n_cat  <- vapply(purity_blocks, `[[`, "n_cat", FUN.VALUE = numeric(1))
purity_order  <- order(-purity_n_cat)
purity_blocks <- purity_blocks[purity_order]
purity_panels <- lapply(purity_blocks, `[[`, "plot")
purity_n_cat  <- purity_n_cat[purity_order]
# Panel AREA (not just linear size) scales with cell-type count, same
# convention as fig2 Panel C.
purity_widths <- sqrt(purity_n_cat)

# Same-scale panels share a row (3 largest together, 3 smallest together).
C_row1_idx <- seq_len(min(3, length(purity_widths)))
C_row2_idx <- setdiff(seq_along(purity_widths), C_row1_idx)

options(repr.plot.width = 20, repr.plot.height = 4)
print(wrap_plots(purity_panels, ncol = 6, guides = "collect", widths = purity_widths))


## Panel D: per-dataset LV x cell-type correlation heatmaps

In [ ]:
corr_full   <- fread(snakemake@input[["corr_full"]])
assignments <- fread(snakemake@input[["assignments"]])
quality_by_dataset <- fread(snakemake@input[["quality_by_dataset"]])

DATASETS_D <- names(DATASET_LABELS)

corr_blocks <- lapply(DATASETS_D, function(ds) {
  d        <- corr_full[dataset == ds & !is.na(cell_type) & cell_type != "NA"]
  assigned <- assignments[dataset == ds & !is.na(cell_type) & cell_type != "NA"]
  if (nrow(d) == 0 || nrow(assigned) == 0) return(NULL)
  # Restrict to the selected (assigned) LV x cell-type pairs -- matches the
  # dot plot and prevents unassigned cell types from collapsing into an
  # artificial NA factor level.
  d <- d[LV %in% assigned$LV & cell_type %in% assigned$cell_type]

  ord      <- order(assigned$cell_type)
  lv_order <- unique(assigned$LV[ord])
  ct_order <- unique(ct_label(assigned$cell_type[ord]))

  d[, cell_type_label := factor(ct_label(cell_type), levels = ct_order)]
  d[, LV              := factor(LV, levels = lv_order)]

  q         <- quality_by_dataset[dataset == ds]
  tau_row   <- q[metric == "mean_tau"]
  auroc_row <- q[metric == "mean_auroc"]
  tau_txt   <- if (nrow(tau_row))   sprintf("mean Tau = %.3f ± %.3f", tau_row$mean, tau_row$sd) else NA_character_
  auroc_txt <- if (nrow(auroc_row)) sprintf("mean AUROC = %.3f ± %.3f", auroc_row$mean, auroc_row$sd) else NA_character_

  # A subtitle (not corner-annotate() text) -- annotate() at a fixed vjust
  # doesn't scale with panel height, so it collided with the topmost data
  # row on tall/short panels alike. Same clean subtitle convention as Panel C.
  subtitle_txt <- if (!is.na(tau_txt)) paste(tau_txt, auroc_txt, sep = "\n") else ""

  p <- ggplot(d, aes(x = cell_type_label, y = LV, fill = cor)) +
    geom_tile(color = "white", linewidth = 0.4) +
    geom_text(aes(label = sprintf("%.2f", cor)), size = 2.2) +
    scale_fill_gradientn(colours = DIVERGING_COLORS, values = DIVERGING_VALUES,
                        limits = c(-1, 1), name = "r") +
    coord_fixed() +
    theme_bw(base_size = 7.5) +
    theme(axis.text.x   = element_text(angle = 45, hjust = 1, vjust = 1, size = 6),
          axis.text.y   = element_text(size = 6.5),
          plot.title    = element_text(face = "bold", size = 8, hjust = 0.5),
          plot.subtitle = element_text(size = 5.5, hjust = 0.5, face = "italic", lineheight = 0.9)) +
    labs(x = NULL, y = NULL, title = DATASET_LABELS[ds], subtitle = subtitle_txt)

  list(plot = p, n_cat = max(length(lv_order), length(ct_order)))
})
names(corr_blocks) <- DATASETS_D
corr_blocks <- Filter(Negate(is.null), corr_blocks)

corr_n_cat  <- vapply(corr_blocks, `[[`, "n_cat", FUN.VALUE = numeric(1))
corr_order  <- order(-corr_n_cat)
corr_blocks <- corr_blocks[corr_order]
corr_panels <- lapply(corr_blocks, `[[`, "plot")
corr_n_cat  <- corr_n_cat[corr_order]
corr_widths <- sqrt(corr_n_cat)

D_row1_idx <- seq_len(min(3, length(corr_widths)))
D_row2_idx <- setdiff(seq_along(corr_widths), D_row1_idx)

options(repr.plot.width = 20, repr.plot.height = 4)
print(wrap_plots(corr_panels, ncol = 6, guides = "collect", widths = corr_widths))


## Panel E: zoomed correlations for hard-to-distinguish pairs (shares Panel D's tag)

In [ ]:
related_corr <- fread(snakemake@input[["related_corr"]])

group_corr_panels <- lapply(split(related_corr, related_corr$group_id), function(d) {
  # Columns = each group member's own assigned LV; rows = every member's
  # raw cell type (same set) -- a genuine square comparison matrix. Members
  # from different datasets (an LV was never computed in a different
  # model's LV space) simply have no row for that (row, col) combination,
  # which geom_tile leaves blank -- same as the original notebook.
  members    <- unique(d$assigned_cell_type)
  member_lv  <- d$LV[match(members, d$assigned_cell_type)]
  # hard_corr_long (04_hard_cell_types.ipynb) scopes its "candidates" list
  # by dataset only, not by group_id -- a dataset shared by two different
  # groups (e.g. Brain_Xiong2023 in both the OPC/Oligodendrocyte group and
  # the cross-dataset Inhibitory/Excitatory group) leaks the OTHER group's
  # cell type in as a spurious extra row. Drop anything outside this
  # group's own 2 (or more) legitimate members.
  d <- d[cell_type %in% members]

  row_levels <- wrap_label(abbreviate_ct(ct_label(members)), width = 14)
  col_levels <- wrap_label(abbreviate_ct(paste0(ct_label(members), " - ", member_lv)), width = 14)

  d[, row_label := factor(wrap_label(abbreviate_ct(ct_label(cell_type)), width = 14), levels = row_levels)]
  d[, col_label := factor(wrap_label(abbreviate_ct(paste0(ct_label(assigned_cell_type), " - ", LV)), width = 14), levels = col_levels)]

  panel_title <- paste(strwrap(paste(abbreviate_ct(ct_label(members)), collapse = " vs. "), width = 25), collapse = "\n")

  # Plain Pearson r (same DIVERGING_COLORS/limits as Panel D, not the
  # dot-grid's row_effect, which mixes in a one-vs-rest effect size that
  # can exceed +/-1) -- lets D and E share one legend.
  ggplot(d, aes(x = col_label, y = row_label, fill = cor)) +
    geom_tile(color = "white", linewidth = 0.5) +
    geom_text(aes(label = sprintf("%.2f", cor)), size = 2.6) +
    scale_fill_gradientn(colours = DIVERGING_COLORS, values = DIVERGING_VALUES,
                        limits = c(-1, 1), name = "r") +
    coord_fixed() +
    theme_bw(base_size = 7.5) +
    theme(axis.text.x = element_text(angle = 45, hjust = 1, size = 6.5),
          axis.text.y = element_text(size = 6.5),
          axis.title  = element_blank(),
          plot.title  = element_text(face = "bold", size = 7, hjust = 0.5,
                                      lineheight = 0.95, margin = margin(b = 2)),
          plot.margin = margin(1, 1, 1, 1)) +
    labs(x = NULL, y = NULL, title = panel_title)
})

options(repr.plot.width = 20, repr.plot.height = 6)
print(wrap_plots(group_corr_panels, ncol = 2, guides = "collect"))


## Panel F: GTEx tissue-clustering benchmark (ARI)

In [ ]:
ari_data        <- fread(snakemake@input[["ari_data"]])
ari_comparisons <- fread(snakemake@input[["ari_comparisons"]])

mean_order_ari <- ari_data[, .(m = mean(ari, na.rm = TRUE)), by = method]
setorder(mean_order_ari, -m)
method_order_ari <- as.character(mean_order_ari$method)
ari_data[, method := factor(method, levels = method_order_ari)]

METHOD_COLORS_ARI <- unlist(cfg$MODEL_COLORS)[method_order_ari]
METHOD_COLORS_ARI[is.na(METHOD_COLORS_ARI)] <- "grey70"
names(METHOD_COLORS_ARI) <- method_order_ari

mean_df_ari <- ari_data[, .(mean_ari = mean(ari, na.rm = TRUE),
                            max_ari  = max(ari, na.rm = TRUE)), by = method]
mean_df_ari[, method := factor(method, levels = method_order_ari)]

xpos_ari <- setNames(seq_along(method_order_ari), method_order_ari)

fmt_q_ari <- function(q) {
  if (q >= 0.001) {
    sprintf('"%.3f"', q)
  } else {
    e_str    <- formatC(q, format = "e", digits = 1)
    parts    <- strsplit(e_str, "e")[[1]]
    mantissa <- trimws(parts[1])
    exp_val  <- as.integer(parts[2])
    sprintf('%s%%*%%10^{%d}', mantissa, exp_val)
  }
}

# Comparisons already computed upstream (kmeans_clustering.py's one-sided
# Mann-Whitney U + BH adjustment, see 00_kmeans_clustering.ipynb) -- reused
# directly rather than recomputing the significance test here.
assign_bracket_tiers_ari <- function(comp_df, xpos) {
  comp_ord <- copy(comp_df)
  comp_ord[, x1 := xpos[a]]
  comp_ord[, x2 := xpos[b]]
  comp_ord[, left  := pmin(x1, x2)]
  comp_ord[, right := pmax(x1, x2)]
  comp_ord[, span  := right - left]
  setorder(comp_ord, span, q)

  levels_used <- list()
  comp_ord[, tier := 0L]

  for (i in seq_len(nrow(comp_ord))) {
    left  <- comp_ord$left[i]
    right <- comp_ord$right[i]
    tier  <- 1

    repeat {
      current <- if (tier <= length(levels_used)) levels_used[[tier]] else NULL
      overlaps <- FALSE

      if (!is.null(current)) {
        overlaps <- any(vapply(current, function(interval) {
          !(right < interval[1] || left > interval[2])
        }, logical(1)))
      }

      if (!overlaps) break
      tier <- tier + 1
    }

    comp_ord$tier[i] <- tier
    prior <- if (tier <= length(levels_used)) levels_used[[tier]] else list()
    levels_used[[tier]] <- c(prior, list(c(left, right)))
  }

  comp_ord
}

comp_ord_ari <- assign_bracket_tiers_ari(ari_comparisons, xpos_ari)
n_tiers_ari <- max(comp_ord_ari$tier)
y_base_ari  <- 1.08
y_step_ari  <- 0.17
h_ari       <- 0.02
y_max_ari   <- y_base_ari + (n_tiers_ari - 1) * y_step_ari + 0.14

plot_F <- ggplot(ari_data, aes(method, ari, fill = method)) +
  geom_boxplot(width = 0.5, outlier.shape = NA, color = "black",
               linewidth = 0.25, alpha = 0.85) +
  geom_jitter(width = 0.10, size = 0.35, shape = 21, fill = "white",
              color = "#333333", stroke = 0.15, alpha = 0.75) +
  geom_point(data = mean_df_ari, aes(x = method, y = mean_ari), shape = 23,
             size = 1.4, fill = "white", color = "black", stroke = 0.4,
             inherit.aes = FALSE) +
  geom_text(data = mean_df_ari,
            aes(x = method, y = max_ari + 0.05, label = sprintf("%.3f", mean_ari)),
            size = 3.2, fontface = "bold", inherit.aes = FALSE) +
  scale_fill_manual(values = METHOD_COLORS_ARI, na.value = "grey70") +
  scale_y_continuous(breaks = seq(0, 1, 0.25), expand = expansion(mult = c(0.02, 0.02))) +
  coord_cartesian(ylim = c(0, y_max_ari), clip = "on") +
  labs(x = NULL, y = "Adjusted Rand Index\n(GTEx tissue-label concordance)") +
  theme_pub() +
  theme(legend.position = "none",
        axis.text.x = element_text(angle = 35, hjust = 1, size = AXIS_LABEL_SIZE),
        panel.grid.major = element_line(color = "grey88", linewidth = 0.25),
        panel.grid.minor = element_blank())

for (i in seq_len(nrow(comp_ord_ari))) {
  row <- comp_ord_ari[i, ]
  y   <- y_base_ari + (row$tier - 1) * y_step_ari
  lbl <- fmt_q_ari(row$q)

  plot_F <- plot_F +
    annotate("segment", x = row$left,  xend = row$left,  y = y,      yend = y + h_ari, linewidth = 0.25) +
    annotate("segment", x = row$left,  xend = row$right, y = y + h_ari, yend = y + h_ari, linewidth = 0.25) +
    annotate("segment", x = row$right, xend = row$right, y = y,      yend = y + h_ari, linewidth = 0.25) +
    annotate("text",
             x = (row$left + row$right) / 2,
             y = y + h_ari + 0.015,
             label = lbl,
             size = 3.0,
             vjust = 0,
             parse = TRUE,
             family = "sans")
}

options(repr.plot.width = 6, repr.plot.height = 3.5)
print(plot_F)

## Panel G: tissue- and subtissue-level B-matrix concordance

Left: per-tissue concordance (% of the rank-1 SHAP-selected LV's top-1%
highest-activity samples that belong to the true tissue, from
`02_b_matrix.ipynb`). Right: donor-grouped out-of-fold subtissue confusion
detail for Heart (2 subtissues) and Brain (13 subtissues, hardest case)
from the subtissue-recovery SHAP-derived logistic regression
(`04_subtissues.ipynb` / `subtissue_lr_eval_gtex`).

In [ ]:
tissue_concordance <- fread(snakemake@input[["tissue_concordance"]])
setorder(tissue_concordance, Pct_Correct)
tissue_concordance[, Tissue := factor(Tissue, levels = Tissue)]

plot_G_left <- ggplot(tissue_concordance, aes(x = Tissue, y = Pct_Correct, fill = Pct_Correct >= 50)) +
  geom_col(width = 0.7) +
  geom_hline(yintercept = 50, linetype = "22", color = "grey40", linewidth = 0.3) +
  coord_flip() +
  scale_fill_manual(values = c(`TRUE` = "#007a33", `FALSE` = "#b2182b"), guide = "none") +
  scale_y_continuous(limits = c(0, 100), expand = expansion(mult = c(0, 0.02))) +
  labs(x = NULL, y = "% of top-LV top-1% samples\nmatching true tissue") +
  theme_pub() +
  theme(axis.text.y = element_text(size = 5.5))

subtissue_confusion <- fread(snakemake@input[["subtissue_confusion"]])

make_confusion_plot <- function(tissue_name) {
  d <- subtissue_confusion[Tissue == tissue_name & Model == "SHAP_LR"]
  lvls <- sort(unique(c(d$True_SMTSD, d$Predicted_SMTSD)))
  d[, True_SMTSD      := factor(True_SMTSD, levels = lvls)]
  d[, Predicted_SMTSD := factor(Predicted_SMTSD, levels = lvls)]

  ggplot(d, aes(x = Predicted_SMTSD, y = True_SMTSD, fill = Count)) +
    geom_tile(color = "white", linewidth = 0.4) +
    geom_text(aes(label = Count), size = 2.2) +
    scale_fill_gradientn(colours = GREEN_SCALE, name = "N samples") +
    coord_fixed() +
    theme_bw(base_size = 7) +
    theme(axis.text.x = element_text(angle = 45, hjust = 1, size = 5.5),
          axis.text.y = element_text(size = 5.5),
          plot.title  = element_text(face = "bold", size = 7.5, hjust = 0.5)) +
    labs(x = "Predicted subtissue (out-of-fold)", y = "True subtissue", title = tissue_name)
}

plot_G_heart <- make_confusion_plot("Heart")
plot_G_brain <- make_confusion_plot("Brain")

plot_G_right <- wrap_plots(plot_G_heart, plot_G_brain, ncol = 1, heights = c(1, 3)) &
  theme(legend.position = "none")

plot_G <- plot_grid(plot_G_left, plot_G_right, ncol = 2, rel_widths = c(1, 1.1))

options(repr.plot.width = 12, repr.plot.height = 10)
print(plot_G)

## Assembly

Row 1: A + B. Row 2: C (single tag, top-anchored 2-row grid sized by
cell-type count, same mechanism as fig2 Panel C). Row 3: D + E, sharing a
single "D" tag -- E is a zoomed detail of specific cell-type pairs already
shown in D, exactly mirroring fig2's C/D relationship (top-anchored rows +
zoom-connector + one shared legend).

In [ ]:
# A single, fixed mm-per-sqrt(n_cat) unit applies to every dataset in both
# C and D, regardless of which row they land in -- panel size vs.
# cell-type count is the same ratio everywhere, matching fig2's convention.
UNIT_PER_SQRT_NCAT <- 28.5

size_row_geometry <- function(widths, row1_idx, row2_idx) {
  sum1 <- sum(widths[row1_idx]); max1 <- max(widths[row1_idx])
  sum2 <- sum(widths[row2_idx]); max2 <- max(widths[row2_idx])
  block_width <- UNIT_PER_SQRT_NCAT * sum1
  row1_h      <- UNIT_PER_SQRT_NCAT * max1
  row2_h      <- UNIT_PER_SQRT_NCAT * max2
  list(width = block_width, row1_h = row1_h, row2_h = row2_h,
       height = row1_h + row2_h, row2_pad = sum1 - sum2)
}

C_geom <- size_row_geometry(purity_widths, C_row1_idx, C_row2_idx)
D_geom <- size_row_geometry(corr_widths,   D_row1_idx, D_row2_idx)

C_width <- C_geom$width
D_width <- D_geom$width
H_C     <- C_geom$height
H_D     <- D_geom$height

DE_GAP <- 14
# E: 2 cols x 3 rows of uniformly-sized square panels, sized so its total
# height matches H_D -- a 2-wide grid needs width:height = 2:3 for square
# cells, same formula fig2 uses for its own zoomed-detail block.
E_width <- H_D * (2 / 3)

# Row 3 (D + E) is the widest row; row 2 (C alone) is centered within the
# same total FIG_W.
FIG_W <- D_width + DE_GAP + E_width

# Row 1 (A + B): simple side-by-side split, same height (no square-grid
# constraint on B here, unlike fig2 -- Panel B is a plain bar chart).
A_width <- FIG_W * 0.55
B_width <- FIG_W * 0.45
ROW1_H  <- 78

# Top-align panels of different sizes within a row -- literal vertical
# split via cowplot::plot_grid(ncol = 1) so panels reliably start flush at
# the same top edge (the patchwork area()-based t/b span trick looked
# equivalent on paper but did not actually keep titles aligned once row
# weights got far apart -- see fig2-assemble).
make_top_anchored_row <- function(panels, weights, pad_weight = 0) {
  row_max <- max(weights)
  cells <- lapply(seq_along(panels), function(i) {
    p <- panels[[i]] + theme(legend.position = "none")
    if (weights[i] < row_max) {
      plot_grid(p, NULL, ncol = 1,
                rel_heights = c(weights[i], row_max - weights[i]))
    } else {
      p
    }
  })
  if (pad_weight > 0) {
    cells   <- c(cells, list(NULL))
    weights <- c(weights, pad_weight)
  }
  plot_grid(plotlist = cells, nrow = 1, rel_widths = weights)
}

C_row1 <- make_top_anchored_row(purity_panels[C_row1_idx], purity_widths[C_row1_idx])
C_row2 <- make_top_anchored_row(purity_panels[C_row2_idx], purity_widths[C_row2_idx], pad_weight = C_geom$row2_pad)
C_block <- plot_grid(C_row1, C_row2, ncol = 1, rel_heights = c(C_geom$row1_h, C_geom$row2_h))

D_row1 <- make_top_anchored_row(corr_panels[D_row1_idx], corr_widths[D_row1_idx])
D_row2 <- make_top_anchored_row(corr_panels[D_row2_idx], corr_widths[D_row2_idx], pad_weight = D_geom$row2_pad)
D_block <- plot_grid(D_row1, D_row2, ncol = 1, rel_heights = c(D_geom$row1_h, D_geom$row2_h))

E_block <- wrap_plots(group_corr_panels, ncol = 2) &
    theme(legend.position = "none")

# Zoom-in connector in the D/E gap: two lines converging from the vertical
# middle of D's edge out to E's top-left/bottom-left corners -- same visual
# fig2 uses between its C and D blocks.
zoom_connector <- ggplot() +
  geom_segment(aes(x = 0, xend = 1, y = 0.5, yend = 1),
               linewidth = 0.4, color = "grey50", linetype = "22") +
  geom_segment(aes(x = 0, xend = 1, y = 0.5, yend = 0),
               linewidth = 0.4, color = "grey50", linetype = "22") +
  scale_x_continuous(limits = c(0, 1), expand = c(0, 0)) +
  scale_y_continuous(limits = c(0, 1), expand = c(0, 0)) +
  theme_void()

row_DE <- plot_grid(
    D_block, zoom_connector, E_block,
    ncol = 3, rel_widths = c(D_width, DE_GAP, E_width),
    labels = c("D", "", ""),
    label_size = LABEL_SIZE, label_fontface = LABEL_FONTFACE,
    label_fontfamily = LABEL_FONTFAMILY
)

# C's own purity legend, centered directly under C_block (not the shared
# bottom row -- D and E share one "r" scale, but C's purity (%) scale is
# unrelated to it).
legend_C_grob <- cowplot::get_legend(
  purity_panels[[1]] +
    theme(legend.position = "top", legend.direction = "horizontal",
          legend.key.size = unit(3.2, "mm"),
          legend.text = element_text(size = LEGEND_TEXT_SIZE + 1.5),
          legend.title = element_text(size = LEGEND_TITLE_SIZE + 1.5))
)
C_LEGEND_H <- 12
C_with_legend <- plot_grid(
    C_block, ggdraw() + draw_grob(legend_C_grob),
    ncol = 1, rel_heights = c(H_C, C_LEGEND_H)
)

# C alone, centered within the same FIG_W row_DE occupies.
C_pad  <- (FIG_W - C_width) / 2
row_C  <- plot_grid(
    NULL, C_with_legend, NULL,
    ncol = 3, rel_widths = c(C_pad, C_width, C_pad),
    labels = c("", "C", ""),
    label_size = LABEL_SIZE, label_fontface = LABEL_FONTFACE,
    label_fontfamily = LABEL_FONTFAMILY
)

row1 <- plot_grid(
    plot_A, plot_B,
    ncol = 2, rel_widths = c(A_width, B_width),
    labels = c("A", "B"),
    label_size = LABEL_SIZE, label_fontface = LABEL_FONTFACE,
    label_fontfamily = LABEL_FONTFAMILY
)

# D and E now share the same correlation scale (Panel E uses the plain
# Pearson r from related_lv_correlations.csv, not the dot-grid's
# row_effect) -- one shared legend, centered on the full page width.
legend_D_grob <- cowplot::get_legend(
  corr_panels[[1]] +
    theme(legend.position = "top", legend.direction = "horizontal",
          legend.key.size = unit(3.2, "mm"),
          legend.text = element_text(size = LEGEND_TEXT_SIZE + 1.5),
          legend.title = element_text(size = LEGEND_TITLE_SIZE + 1.5))
)
LEGEND_H     <- 14
legend_width <- 0.3
legend_pad   <- (1 - legend_width) / 2
legend_row <- plot_grid(NULL, ggdraw() + draw_grob(legend_D_grob), NULL, nrow = 1,
                        rel_widths = c(legend_pad, legend_width, legend_pad))

# Row 4 (F + G): GTEx-derived panels, kept as their own row below the
# pseudobulk-only A-E rows. G is itself a 2-part composite (tissue
# concordance bars + subtissue confusion detail), so it needs more width
# than F -- weighted 0.35/0.65 rather than F+G's row1-style 0.55/0.45.
# Height is set by G's 49-tissue bar list (the tallest element in the row);
# F just gets extra headroom, matching how row_C is centered/padded rather
# than tightly fit elsewhere in this notebook.
F_width  <- FIG_W * 0.35
G_width  <- FIG_W * 0.65
ROW_FG_H <- 165

row_FG <- plot_grid(
    plot_F, plot_G,
    ncol = 2, rel_widths = c(F_width, G_width), align = "h", axis = "tb",
    labels = c("F", "G"),
    label_size = LABEL_SIZE, label_fontface = LABEL_FONTFACE,
    label_fontfamily = LABEL_FONTFAMILY
)

FIG_H <- ROW1_H + H_C + C_LEGEND_H + H_D + LEGEND_H + ROW_FG_H

supp1 <- plot_grid(
    row1, row_C, row_DE, legend_row, row_FG,
    ncol = 1, rel_heights = c(ROW1_H, H_C + C_LEGEND_H, H_D, LEGEND_H, ROW_FG_H)
)

ggsave(snakemake@output[["pdf"]], supp1,
       width = FIG_W, height = FIG_H, units = "mm",
       device = cairo_pdf, bg = "white")
ggsave(snakemake@output[["png"]], supp1,
       width = FIG_W, height = FIG_H, units = "mm",
       dpi = 300, bg = "white")
ggsave(snakemake@output[["svg"]], supp1,
       width = FIG_W, height = FIG_H, units = "mm",
       device = svglite::svglite, bg = "white")
cat("supp1 exported to", dirname(snakemake@output[["png"]]), "\n")
cat("FIG_W =", FIG_W, "mm, FIG_H =", FIG_H, "mm, C_width =", C_width, "D_width =", D_width, "E_width =", E_width, "\n")
supp1
